## Streamlined Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
from shapely.geometry import Polygon
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [ ]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [2]:
# set a bbox for investigation
bbox = box(-117.161615,32.703371,-117.129171,32.720505)

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
   "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [32]:
parcels.head()

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry
0,2886921500,San Diego,None,None,None,184.829489,2010.093529,"MULTIPOLYGON Z (((-116.81690 33.01572 0.00000,..."
1,2892402500,San Diego,None,None,None,285.581225,4578.667349,"MULTIPOLYGON Z (((-116.61986 33.04945 0.00000,..."
2,2886826141,San Diego,None,None,None,43.395786,107.126775,"MULTIPOLYGON Z (((-116.78047 33.00518 0.00000,..."
3,2982130831,San Diego,None,None,None,397.588311,8452.894325,"MULTIPOLYGON Z (((-117.27009 32.98435 0.00000,..."
4,2982131510,San Diego,None,None,None,397.588311,8452.894325,"MULTIPOLYGON Z (((-117.27009 32.98435 0.00000,..."


In [4]:
# import Zillow data (make take 10-20 minutes)
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint
There are a total of 8 parquet files that correspond to California. We select the one that contain Los Angeles county (w120_n35_w115_n30.parquet).

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Streamlined Code (takes 1 minute!)

In [25]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to multi-family home residential parcels
# keep the parcels where multi-family homes match to parcels
parcels_res = parcels.sjoin(zillow_multi, predicate="contains")

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

In [33]:
# keep only the necessary columns
parcels_res = parcels_res[['PARNO', 'County', 'geometry', 'type', 'year', 'room', 'heat', 'cool', 'own', 'unit', 'value', 'sqft_type', 'sqft', 'GEOID', 'area']]

In [ ]:
parcels_res

Change the code to keep the `PARNO` (parcel number with each of the buildings).

In [35]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects")

# drop the right index
valid_buildings = valid_buildings.drop(columns = 'index_right')

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)


In [36]:
valid_buildings.head()

,source,id,height,var,region,bbox,geometry,PARNO,County,type,...,room,heat,cool,own,unit,value,sqft_type,sqft,GEOID,area
6344544,osm,546629542,2.095237,0.773937,USA,"{'xmax': -116.79879449999999, 'xmin': -116.799...","POLYGON ((-116.79891 33.39104, -116.79901 33.3...",1131300900,San Diego,Multi,...,NaN,None,None,None,2.0,693124.0,living,5640.0,06073020903,SDGE
6344637,osm,546630246,2.132655,0.123639,USA,"{'xmax': -116.7952663, 'xmin': -116.7954647999...","POLYGON ((-116.79539 33.39013, -116.79546 33.3...",1131300900,San Diego,Multi,...,NaN,None,None,None,2.0,693124.0,living,5640.0,06073020903,SDGE
6344638,ms,UnitedStates_023013203_218928,8.093865,2.164009,USA,"{'xmax': -116.79542502150233, 'xmin': -116.795...","POLYGON ((-116.79555 33.39021, -116.79543 33.3...",1131300900,San Diego,Multi,...,NaN,None,None,None,2.0,693124.0,living,5640.0,06073020903,SDGE
6344639,ms,UnitedStates_023013203_277953,6.538584,2.185169,USA,"{'xmax': -116.7951542354982, 'xmin': -116.7952...","POLYGON ((-116.79529 33.38950, -116.79524 33.3...",1131300900,San Diego,Multi,...,NaN,None,None,None,2.0,693124.0,living,5640.0,06073020903,SDGE
6344640,ms,UnitedStates_023013203_277952,6.395244,3.757385,USA,"{'xmax': -116.79521751197618, 'xmin': -116.795...","POLYGON ((-116.79522 33.38880, -116.79522 33.3...",1131300900,San Diego,Multi,...,NaN,None,None,None,2.0,693124.0,living,5640.0,06073020903,SDGE


In [37]:
## CREATE CENTROIDS FOR EACH MULTI-RESIDENTIAL BUILDINGS
# change the crs to a projected crs as centroids only works in projected crs
valid_buildings = valid_buildings.to_crs('EPSG:3857')

# create centroids for each multi residential builing
valid_building_centroids = valid_buildings.centroid

## join the centroids with the valid buildings to keep the additional building information 
# select only the necessary columns of the dataframe
valid_building_ids = valid_buildings[['source', 'id', 'height', 'var', 'PARNO']]

# make the centroids a dataframe for the join
valid_building_centroids = pd.DataFrame(valid_building_centroids)

# join the centroids with the building ids
multi_buildings = valid_building_ids.join(valid_building_centroids)


In [ ]:
multi_buildings.head()

,source,id,height,var,PARNO,0
6344544,osm,546629542,2.095237,0.773937,1131300900,POINT (-13001994.977 3947309.735)
6344637,osm,546630246,2.132655,0.123639,1131300900,POINT (-13001600.622 3947191.971)
6344638,ms,UnitedStates_023013203_218928,8.093865,2.164009,1131300900,POINT (-13001619.320 3947225.199)
6344639,ms,UnitedStates_023013203_277953,6.538584,2.185169,1131300900,POINT (-13001584.417 3947114.213)
6344640,ms,UnitedStates_023013203_277952,6.395244,3.757385,1131300900,POINT (-13001588.313 3947036.571)


In [ ]:
# create column from polygon area
multi_buildings['area_m2'] = multi_buildings.geometry.area

# rename height column to be clear about units
multi_buildings.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
multi_buildings['volume_m3'] = multi_buildings['area_m2'] * multi_buildings['height_m']

# 6. keep only observations with unit data
building_w_units = multi_buildings[~multi_buildings['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

In [ ]:
building_w_units.head()

In [ ]:
# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]


In [ ]:
print(f"The intercept is ", intercept)
print(f"The slope is ", slope)

In [ ]:
# ## join parcels to buildings (keeping observations as parcels, but with building attributes)
# # sum number of units per parcel
# units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# # join on parcels with summed number of units
# parcels_res = parcels_res.join(units_sum)


# # keep all residential buildings, and add zillow points only where they match up
# building_zillow = gpd.sjoin(
#     buildings_res,
#     zillow_multi,
#     how = "left",
#     predicate = "intersects")

# # reproject data frame to crs with meters as units
# building_m = building_zillow.to_crs("EPSG:6933")

# # create column from polygon area
# building_m['area_m2'] = building_m.geometry.area

# # rename height column to be clear about units
# building_m.rename(columns={"height":"height_m"}, inplace = True)

# # create volume column
# building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# # 6. keep only observations with unit data
# building_w_units = building_m[~building_m['unit'].isna()]

# assert building_w_units['unit'].isna().sum() == 0

# # run model
# results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# # add residuals as a column
# building_w_units['residual'] = results.resid.copy()

# # keep only observations that are less/equal to 2 standard deviations from residuals
# building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# # save outliers, as we will re-predict them using the regression
# building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# # rerun linear regression
# results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# # save variables
# intercept = results_clean.params[0]
# slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual columns
multi_complete = multi_complete.drop(['residual'], axis = 1)

multi_complete = multi_complete.drop(['index_right'], axis = 1)

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# join back to parcels
units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# join on parcels with summed number of units
multi_summed_units = parcels_res.join(units_multi)

assert len(multi_summed_units) < len(multi_by_parcel)

# save all non-multi observations
non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)

# keep only variables of interest
non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

## Final results

In [ ]:
multi_complete.head() # should be my buildings??

In [ ]:
multi_summed_units.head(3)

In [ ]:
multi_by_parcel.head(3)

In [ ]:
non_multi_points.head(3)

## Renaming and saving

In [ ]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [ ]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

In [ ]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [ ]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [ ]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')